In [ ]:
# Autotamic reload of source files
%load_ext autoreload
%autoreload 2
import os
import sys
import matplotlib.pyplot as plt

# Get the current working directory
current_dir = os.path.dirname(os.path.abspath('__file__'))

# Construct the path to the project root directory
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# Add the project root directory to the Python path
if project_root not in sys.path:
    sys.path.append(project_root)

# Import dataLoader
from src.data_processing import DataLoader
# Import the Metrics class
from src.data_processing import Metrics
# Import plotter
from src.plotting_general import *

data = DataLoader().import_all_data()
print(data.keys())

participants = data["user_interactions"]["participant_id"].unique()
participants.sort()
print("Participants:", participants)

metrics = Metrics(data)

mapping_questions = {
    "Understand predictions": "After completing the experiment, I understand the reasoning behind the classifier's predictions.",
    "Alignment with expectations": "The predictions of the classifier align with my understanding of machine learning image classification.",
    "Reliable predictions": "I feel that the predictions provided by the satellite image classifier are reliable.",
    "Trust retrained model": "I would trust the retrained satellite image classifier more if it were updated based on the test set I created.",
    "Unpredictable reaction": "The system reacts unpredictably.",
    "Ability to understand": "I was able to understand why the image classifier made mistakes.",
    "Real world app trust": "I would trust the system if it was deployed in a real-world application.",
    "Accurate classifier": "Overall, the image classifier is accurate.",
    "Approach to test set creation": "Which of the following best describes your strategy for creating your test set?", # ! Important
    "Strategy: [01]": "Please briefly explain your strategy for creating your test set, if any:", # ! Important
    "Factors influencing prioritization": "Which of the following factors mostly influenced your choice of which classes to prioritize, as the testing progressed?", # ! Important
    'Surprising discrepancies': "Was there anything in the behavior of the classifier that you found surprising?", # ! might be important
    "Surprising discrepancies: Yes (please explain what were your surprises)": "Was there anything in the behavior of the classifier that you found surprising?", # ! might be important
    "Gender": "Which gender do you identify with?",
    "Rate expertise": "How would you rate your expertise in machine learning?",
    "Age: [01]": "How old are you?",
}

# Specify participant list and ordered categories
pid_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
ordered_categories = ["Strongly Disagree", "Disagree", "Neither Agree nor Disagree", "Agree", "Strongly Agree"]


In [ ]:
metrics.data["questionnaire"].columns

# Demographics

In [ ]:
# Gender
print(data["questionnaire"]["Gender"].groupby(data["questionnaire"]["Gender"]).count())
# Age
print(data["questionnaire"]["Age: [01]"].groupby(data["questionnaire"]["Age: [01]"]).count())
# Rate expertise
print(data["questionnaire"]["Rate expertise"].groupby(data["questionnaire"]["Rate expertise"]).count())

# Self-reported strategy

In [ ]:
metrics.data["questionnaire"]["Approach to test set creation"].groupby(metrics.data["questionnaire"]["Approach to test set creation"]).count()
# Last item never selected is: "I selected images without a specific strategy.""

In [ ]:
for i, row in enumerate(metrics.data["questionnaire"]["Strategy: [01]"]):
    print(f"{i+1}: {row}", end="\n\n")


1: nan

**2: I first tried to understand the limits of the classifier by selecting challenging images and also proove the system by detecting the images that work well. Then I deleted the too obvious test sets, because I thought its better to you challenging ones. I tried to have test sets from each category.**

3: I tried to find "difficult" images to test the predicition of the algorithm. Sometimes, there were no explicit label or more labels accurate... 

4: nan

**5: i tried to have mainly frames that were challenging as in containing 2 or more of the categories. so for example i would get a road passing through a residential area or a forest, r a bridge going on top of a railway.**

6: i wanted to make the system fails\ so it can learn something new,to be better at somethńg the systems usually get confuse, 

7: I wanted to select challenging sets so that the algorithm can improve. therefore I wanted to create as much errors as possible while still having enough agreeable ones

8: First, I focused on selecting prototypical images for the classifer to assert the success areas of the system, then selected challenging images for the classifier in order to uncover the system fails. Then, I just used the randomizer to select equal proportions of challenging and 'easy' images mainly to increase sample size. 

9: nan

**10: I tried to find the most challenging pictures for the classifier. E.g. residential areas with different building forms and roof colors, different colored water bodies, different types of railroads, ...**

11: I chose images where there were thin separations between 2 adjacent images, hoping it would push the classifier to understand what differentiates one image from another.

**12: First I chose a rather random set of images to get an impression of how the system works in general.
Secondly, I chose some more prototypical examples for the categories which did not work in the first place.
When I started to run out of time, i just added some more (and more various) examples for the categories which seemed to be difficult for the model**
...
**14: I changed my strategy a few times during the experiment.**

15: I chose challenging images but also a few not challenging. Sometime i chose quiet similar pictures, one failed one not for showing that just a little difference let the system fail. **I noticed that especially public places and parks were not recognized correctly, so I tried to add correctly recognized images and incorrectly recognized images in equal measure. Unfortunately, I could not add a wrongly recognized image to the train tracks, as all images were recognized correctly.**

In [ ]:
print(metrics.data["questionnaire"]["Factors influencing prioritization"].groupby(metrics.data["questionnaire"]["Factors influencing prioritization"]).count())
# Last item never selected is: "The bar chart"

In [ ]:
print(metrics.data["questionnaire"]['Surprising discrepancies'].groupby(metrics.data["questionnaire"]['Surprising discrepancies']).count())
print(metrics.data["questionnaire"]['Surprising discrepancies: Yes (please explain what were your surprises)'].groupby(metrics.data["questionnaire"]['Surprising discrepancies: Yes (please explain what were your surprises)']).count())

In [ ]:
import plot_likert
# Prepare the Likert-scale data
likert_data = pd.DataFrame({
    mapping_questions["Accurate classifier"]: metrics.data["questionnaire"]["Accurate classifier"].dropna().values,
    mapping_questions["Ability to understand"]: metrics.data["questionnaire"]["Ability to understand"].dropna().values,
    mapping_questions["Unpredictable reaction"]: metrics.data["questionnaire"]["Unpredictable reaction"].dropna().values
})

# Ensure Likert-scale responses are categorical with a standard order
ordered_responses = ["Strongly Disagree", "Disagree", "Neither Agree nor Disagree", "Agree", "Strongly Agree"]
for column in likert_data.columns:
    likert_data[column] = pd.Categorical(likert_data[column], categories=ordered_responses, ordered=True)


plot_likert.plot_likert(likert_data, ordered_responses, plot_percentage=False, figsize=(9,1.5))

# Adjust legend position (above the plot)
plt.legend(bbox_to_anchor=(0.5, -0.5), loc='lower center', ncol=5, frameon=False)
plt.xlabel("")

plt.savefig("../figures/lbw/likert_responses.pdf", bbox_inches="tight")